<a href="https://colab.research.google.com/github/rishabh1024/langgraph-colab-notebooks/blob/main/Langchain_Agent_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U --quiet langchain langgraph langchain_openai
!pip install -U --quiet tavily-python
!pip install -U --quiet langchain-tavily
!pip install -U --quiet langchain_openai

In [ ]:
from google.colab import userdata
import os

openrouter_api_key = userdata.get('OPENROUTER_API_KEY')
tavily_api_key = userdata.get('TAVILY_API_KEY')

api_key = userdata.get("LANGSMITH_API_KEY")
if not api_key:
    raise ValueError("No LANGSMITH_API_KEY found in Colab userdata. Please set it first.")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "colab-notebook"
os.environ["LANGCHAIN_API_KEY"] = api_key
os.environ["LANGSMITH_API_KEY"] = api_key
# Replace "my-langchain-project" with the name you want for your project in LangSmith
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [ ]:
import math
from collections import deque
from typing import Optional

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

from pydantic import BaseModel, Field

class Reflection(BaseModel):

  reflections: str = Field(description="The critique and reflections on the sufficiency, superfluency,"
        " and general quality of the response")

  score: int = Field(description="Score from 0-10 on the quality of the candidate response",
                     gte=0,
                     lte=10)

  found_solution: bool = Field(description="Whether the response has fully solved the question or the task")

  def as_message(self):
    return HumanMessage(content=f"Reasoning: {self.reflections} \n Score: = {self.score}")

  def normalized_score(self)->float:
    return self.score / 10.0

In [ ]:
class Node:

  def __init__(self,
               messages: list[BaseMessage],
               reflection: Reflection,
               parent: Optional["Node"]= None):
    self.messages = messages
    self.parent = parent
    self.children = []
    self.value = 0
    self.visits = 0
    self.reflection = reflection
    self.depth = parent.depth + 1 if parent else 1
    self._is_solved = reflection.found_solution if reflection else False
    if self._is_solved:
      self._mark_tree_as_solved()
    # Call the method to get the score
    self.backpropagate(reflection.normalized_score())

  @property
  def __repr__(self) -> str:

    return (
        f"<Node value={self.value}, visits={self.visits},>"
        f" solution={self.messages} reflection {self.reflection}"
    )

  @property
  def is_solved(self):
    return self._is_solved

  @property
  def is_terminal(self):
    return not self.children

  @property
  def best_child_score(self):
    """Return the child with the highest value."""
    if not self.children:
      return None

    return max(self.children, key=lambda child: int(child.is_solved)* child.value)

  @property
  def height(self) -> int:
    """Check for how far we've rolled out the tree."""
    if self.children:
      return 1 + max([child.height for child in self.children])
    return 1

  def upper_confidence_bound(self, exploration_weight=1.0):
    """ Return the UCT score. This helps balance exploration vs exploitation of a branch"""
    if self.parent is None:
      raise ValueError("Cannot obtain UCT score for root node.")
    if self.visits == 0:
      return self.value

    average_reward = self.value / self.visits
    # There was a typo here, parent.vists should be parent.visits
    exploration_term = math.sqrt(math.log(self.parent.visits) / self.visits)

    return average_reward + exploration_weight * exploration_term

  def backpropagate(self, reward: float):

    """ Update the score of this node and its parents"""
    node = self

    while node:
      node.visits += 1
      node.value = (node.value * (node.visits - 1) + reward) / node.visits
      node = node.parent

  def get_messages(self, include_reflections: bool = True) -> list[BaseMessage]:
    if include_reflections:
      return self.messages + [self.reflection.as_message()]
    return self.messages

  def get_trajectory(self, include_reflections: bool = True) -> list[BaseMessage]:
    """ Get messages representing this search branch"""
    messages = []
    node = self

    while node:
      messages.extend(
          node.get_messages(include_reflections=include_reflections)[::-1]
      )
      node = node.parent
    # Reverse the final back-tracked trajectory to return in the correct order
    return messages[::-1] # root soultion, reflection, child 1,...

  def _get_all_children(self):
    all_nodes = []
    nodes = deque()
    nodes.append(self)

    while nodes:
      node = nodes.popleft()
      all_nodes.extend(node.children)
      for n in node.children:
        nodes.append(n)
    return all_nodes

  def get_best_solution(self):

    """Return the best solution from within the current sub-tree """
    all_nodes = [self] + self._get_all_children()
    best_node = max(all_nodes,
                    # We filter out all non-terminal and non-solution trajectories
                    # There was a typo here, is_termninal should be is_terminal
                    key=lambda node: int(node.is_terminal and node.is_solved) * node.value
                    )
    return best_node

  def _mark_tree_as_solved(self):
    node = self
    while node:
      node._is_solved = True
      node = node.parent

In [ ]:
from typing_extensions import TypedDict

class TreeState(TypedDict):
  root: Node
  # original input
  input: str



In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

openrouter_api_key = userdata.get('OPENROUTER_API_KEY')

# Note the model moonshotai/kimi-k2:free inputs are being used for training. Do not share private data.
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_api_key,
    model="openai/gpt-4o-mini", # Using a free model that supports tool use
    extra_body=
    {
        "provider": {
            "order": [
                "z-ai/glm-4.5-air:free",
                "meta-llama/llama-3.3-70b-instruct:free",
                "openai/gpt-oss-20b:free"
            ]
        }
    }
)

In [ ]:
from langchain_tavily import TavilySearch
from langgraph.prebuilt import ToolNode
import os

os.environ["TAVILY_API_KEY"] = tavily_api_key
tavily_search_tool = TavilySearch(api_key=tavily_api_key, max_results=5, search_depth="basic")
tools = [tavily_search_tool]

tool_node = ToolNode(tools=tools)

In [ ]:
from langchain_core.output_parsers.openai_tools import (
    JsonOutputToolsParser,
    PydanticToolsParser
)

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import chain as as_runnable

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "Reflect and grade the assistant's response to the user question below"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="candidate")
    ]
)

reflection_llm_chain = (
    prompt
    |llm.bind_tools(tools = [Reflection], tool_choice="Reflection").with_config(
        run_name="Reflection"
    )
    | PydanticToolsParser(tools=[Reflection])
)

@as_runnable
def reflection_chain(inputs) -> Reflection:
  tool_choices = reflection_llm_chain.invoke(inputs)
  reflection = tool_choices[0]
  if not isinstance(inputs["candidate"][-1], AIMessage):
    reflection.found_solution = False
  return reflection

In [ ]:
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableConfig

prompt_template = ChatPromptTemplate.from_messages(
    [('system',
     'You are an AI Assistant'
     ),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="messages", optional=True),
     ]
)

initial_answer_chain = prompt_template | llm.bind_tools(tools=tools).with_config(
    run_name="GenerateInitialCandidate"
)

parser = JsonOutputToolsParser(return_id=True)

In [ ]:
def generate_initial_response(state: TreeState) -> dict:

  """Generate the initial candidate response"""
  res = initial_answer_chain.invoke({"input": state["input"]})

  parsed = parser.invoke(res)

  tool_responses = [
      tool_node.invoke(
          {
              "messages":[
                  AIMessage(
                            content="",
                            tool_calls=[
                                {"name": r["type"],
                                 "args": r["args"],
                                 "id": r["id"]}
                            ],
                )
              ]
          }
      )
      for r in parsed
  ]

  output_messages = [res] + [tr["messages"][0] for tr in tool_responses]

  reflection = reflection_chain.invoke(
      {"input": state["input"], "candidate": output_messages}
  )

  # Call the method to get the score when creating the Node
  root = Node(output_messages, reflection=reflection)

  return {
      **state,
      "root": root
  }

In [ ]:
# This generates N candidate values
# for a single input to sample actions from the environment

def generate_candidates(messages: ChatPromptValue, config: RunnableConfig):

  n = config["configurable"].get("N", 5)
  bound_kwargs = llm.bind_tools(tools=tools).kwargs

  chat_result = llm.generate(
      [messages.to_messages()],
      n=n,
      callbacks=config["callbacks"],
      run_name="GenerateCandidates",
      **bound_kwargs,
  )
  print(f'Chat Result Gen: {chat_result}')
  return [gen.message for gen in chat_result.generations[0]]

expansion_chain = prompt_template | generate_candidates

In [ ]:
res = expansion_chain.invoke(
    {
        "input": "Write a research report on effects of AI on changing the trends in hiring for engineering roles across the world. Focus on how use of AI by recruiters and candidates is playing the role! Generate multiple queries to get detailed information on the topic"
    }
)

In [ ]:
import json
from langchain_core.messages import AIMessage

# Assuming 'res' is a list of AIMessage objects from cell -gUfItjpXttC
# Convert the list of AIMessage objects to a list of dictionaries using model_dump()
res_dicts = [r.model_dump() for r in res]

# Use json.dumps to pretty print the list of dictionaries
print(json.dumps(res_dicts, indent=4))

In [ ]:
from collections import defaultdict

def select(root: Node) -> dict:
  """Starting from the root node a child node is selected at each tree level until a leaf node is reached"""

  if not root.children:
    return root

  node = root

  while node.children:
    max_child = max(node.children, key=lambda child: child.upper_confidence_bound())
    node = max_child

  return node

def expand(state: TreeState, config: RunnableConfig) -> dict:
  """Starting from the 'best' node in the tree, generate N candidates for the next step"""
  root = state["root"]
  best_candidate: Node = select(root)

  messages = best_candidate.get_trajectory()
  # Generate N candidates from the single child candidate
  new_candidates = expansion_chain.invoke(
      {"input": state["input"], "messages": messages}, config
  )

  parsed = parser.batch(new_candidates)
  flattened = [
      (i, tool_call)
      for i, tool_calls in enumerate(parsed)
      for tool_call in tool_calls
      ]

  tool_responses = [
      (
          i,
          tool_node.invoke(
              {
                  'messages': [
                      AIMessage(
                          content='',
                          tool_calls=[
                            {
                              'name': tool_call['type'],
                              'args': tool_call['args'],
                              'id': tool_call['id']
                            }
                          ]
                      )
                  ]
              }
          )
      )
      for i, tool_call in flattened
  ]

  collected_responses = defaultdict(list)

  for i, resp in tool_responses:
    collected_responses[i].append(resp['messages'][0])
  output_messages = []

  for i, candidate in enumerate(new_candidates):
    output_messages.append([candidate] + collected_responses[i])

  # Reflect on each candidate now
  reflections = reflection_chain.batch(
      [{'input': state['input'], "candidate": msgs} for msgs in output_messages],
      config,
  )

  child_nodes = [
      Node(cand, parent=best_candidate, reflection=reflection)
      for cand, reflection in zip(output_messages, reflections)
  ]

  best_candidate.children.extend(child_nodes)

  return state

In [ ]:
# Create the graph

from typing import Literal

from langgraph.graph import END, StateGraph, START

def should_loop(state:TreeState):
  root = state["root"]
  if root.is_solved:
    return END
  if root.height > 5:
    return END
  return "expand"

builder = StateGraph(TreeState)
builder.add_node("start", generate_initial_response)
builder.add_node("expand", expand)
builder.add_edge(START, "start")

builder.add_conditional_edges(
    "start",
    should_loop,
    ["expand", END],
)

builder.add_conditional_edges(
    "expand",
    should_loop,
    ["expand", END],
)

graph = builder.compile()

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
question = "Generate a table with average size and weight, as well as the oldest recorded instance for each top 5 most common birds"

last_step = None

for step in graph.stream({"input": question}):
  last_step = step
  step_name, step_state = next(iter(step.items()))
  print(f"Step Name: {step_name}")
  print("rolled out: ", step_state["root"].height)
  print("-------")

In [ ]:
solution_node = last_step["expand"]["root"].get_best_solution()
best_trajectory = solution_node.get_trajectory(include_reflections=False)
print(best_trajectory[-1].content)

In [ ]:
question = "Write a case study on the impact of illegal electronic waste dumping on the climate change!"

last_step = None

for step in graph.stream({"input": question}):
  last_step = step
  step_name, step_state = next(iter(step.items()))
  print(f"Step Name: {step_name}")
  print("rolled out: ", step_state["root"].height)
  print("-------")

In [ ]:
solution_node = last_step["expand"]["root"].get_best_solution()
best_trajectory = solution_node.get_trajectory(include_reflections=False)
print(best_trajectory[-1].content)

In [ ]:
from IPython.display import display, Markdown

display(Markdown(best_trajectory[-1].content))

After running this cell, your LangChain runs in subsequent cells should be traced to the specified LangSmith project. You can then view the traces in the LangSmith UI.

# Task
Compare the provided Colab notebook code to the LATS tutorial code at "https://langchain-ai.github.io/langgraph/tutorials/lats/lats/" and highlight the similarities and differences in their implementation of the LATS framework.

## Review the tutorial code

### Subtask:
Analyze the structure and key components of the LATS tutorial code in the provided repository, focusing on how it implements the LATS framework (initial generation, expansion, reflection, tree search logic).


## Review the notebook code

### Subtask:
Examine the relevant cells in your Colab notebook that implement similar functionalities (Node class, TreeState, chains for initial generation, expansion, and reflection, the graph structure).


**Reasoning**:
I will analyze the provided Colab notebook code cells to identify and understand the implementation of the LATS framework components: the `Node` class, `TreeState`, and the chains for initial generation, expansion, and reflection. I will also examine the LangGraph structure to understand how these components are integrated.



In [ ]:
# 1. Node class (Cell: Pes9BMJ1bsmA)
# Analyzing the Node class definition:
# - Stores messages, parent/child relationships.
# - Attributes: messages, parent, children, value, visits, reflection, depth, _is_solved.
# - Methods: __repr__, is_solved, is_terminal, best_child_score, height, upper_confidence_bound, backpropagate, get_messages, get_trajectory, _get_all_children, get_best_solution, _mark_tree_as_solved.
# - It uses 'value' and 'visits' for tree search calculations (e.g., UCB).
# - 'reflection' stores the Reflection object for the node.
# - '_is_solved' indicates if a solution is found in this branch.

# 2. TreeState (Cell: blvhlQzS_iyS)
# Analyzing the TreeState definition:
# - TypedDict holding the state of the LATS tree search.
# - Attributes: 'root' (the root Node of the tree), 'input' (the original user query).

# 3. Initial Generation Chain (Cell: Ecohpm6B7uZG and tBkjZYZ2HDHl)
# Analyzing initial_answer_chain:
# - Defined in cell Ecohpm6B7uZG using ChatPromptTemplate and llm.bind_tools.
# - Takes 'input' and optional 'messages' as variables.
# - Uses the 'llm' (ChatOpenAI) and binds the defined 'tools' (TavilySearch).
# Analyzing generate_initial_response function in cell tBkjZYZ2HDHl:
# - Invokes initial_answer_chain with the input.
# - Parses the tool calls from the LLM response using JsonOutputToolsParser.
# - Executes the tool calls using the tool_node.
# - Combines the initial LLM response and tool responses.
# - Invokes the reflection_chain to get a Reflection for the initial response.
# - Creates the root Node with the generated messages and reflection.

# 4. Expansion Chain (Cell: OxjVUCQojXjB and cMeJpcLE8Y2t)
# Analyzing expansion_chain in cell OxjVUCQojXjB:
# - Defined using ChatPromptTemplate and generate_candidates function.
# - generate_candidates function takes messages and config (including N for number of candidates).
# - Uses llm.generate to produce N candidate AIMessage responses based on the input and previous messages (trajectory).
# Analyzing expand function in cell cMeJpcLE8Y2t:
# - Selects the best candidate node from the current tree using the 'select' function (based on UCB).
# - Gets the trajectory of messages leading to the selected node.
# - Invokes the expansion_chain with the input and trajectory to generate N new candidates.
# - Parses and executes tool calls for each new candidate.
# - Collects the LLM response and tool responses for each candidate.
# - Invokes reflection_chain.batch to get reflections for all N candidates.
# - Creates new child Nodes for the selected best candidate, each representing a new candidate trajectory and its reflection.

# 5. Reflection Chain (Cell: f58443d7)
# Analyzing reflection_chain:
# - Defined using ChatPromptTemplate and llm.bind_tools with the Reflection tool.
# - Takes 'input' and 'candidate' messages as variables.
# - Uses the 'llm' and binds the 'Reflection' Pydantic model as a tool.
# - Uses PydanticToolsParser to parse the LLM's tool call into a Reflection object.
# - Includes a check to set found_solution to False if the candidate's last message is not an AIMessage.

# 6. LangGraph Structure (Cell: fMwpaXoXIaQH)
# Analyzing the graph definition:
# - Uses StateGraph with TreeState.
# - Nodes: 'start' (calls generate_initial_response), 'expand' (calls expand).
# - Edges:
#   - START -> 'start' (initial entry point).
#   - Conditional edge from 'start' based on should_loop: to 'expand' if not solved and height <= 5, else END.
#   - Conditional edge from 'expand' based on should_loop: to 'expand' if not solved and height <= 5, else END.
# - should_loop function:
#   - Checks if the root node is solved (`root.is_solved`).
#   - Checks the height of the tree (`root.height`).
#   - Returns END if solved or height > 5, otherwise returns "expand".
# - The graph iteratively expands the tree until a solution is found or a maximum depth is reached.

## Identify similarities

### Subtask:
Identify the parts of your code that are directly inspired by or closely match the tutorial's approach and components.


**Reasoning**:
Compare the implementation details of the LATS components in the notebook code with the concepts from the LATS tutorial, focusing on identifying direct inspirations and close matches in their approach.



In [ ]:
# 1. Node class comparison:
# The notebook's Node class closely matches the core concepts described in LATS for a tree search node.
# - Attributes for storing messages, parent/children, visit counts ('visits'), and a value ('value') for tracking utility/score are present in both.
# - The inclusion of 'reflection' directly aligns with LATS's emphasis on self-reflection at each step.
# - Key methods like 'upper_confidence_bound' for node selection and 'backpropagate' for updating scores up the tree are central to Monte Carlo Tree Search (MCTS) principles, which LATS builds upon, and are implemented in the notebook.
# - Methods for getting trajectories and identifying the best solution also align with navigating and evaluating the search tree.

# 2. TreeState comparison:
# The TreeState in the notebook is a direct representation of the state needed for the graph, tracking the root of the search tree and the initial input. This aligns with the fundamental requirement in LATS to maintain the state of the ongoing search process, including the current tree structure and the original query.

# 3. Initial Generation comparison:
# The notebook's initial response generation logic (initial_answer_chain and generate_initial_response) mirrors the tutorial's initial generation step.
# - Both use an LLM (llm) to generate a first response to the user's input.
# - Both integrate tool usage (tavily_search_tool) in this initial step, allowing the agent to gather information immediately if needed, which is a common pattern in LATS agents.
# - Critically, both immediately follow the initial generation with a reflection step to evaluate the quality of this first response, creating the root node with this evaluation.

# 4. Expansion Process comparison:
# The expansion process in the notebook (expansion_chain and expand function) is a core implementation of the LATS expansion phase.
# - The 'select' function embodies the tree traversal and node selection logic (specifically using UCB in this MCTS-inspired approach) that is fundamental to deciding which part of the tree to explore next.
# - The use of 'expansion_chain' and llm.generate to produce 'N' candidate continuations from the selected node directly matches the concept of expanding the tree by generating multiple potential next steps.
# - Handling tool usage for these generated candidates and then reflecting on each candidate response aligns perfectly with the iterative cycle of generating possibilities and evaluating them.
# - The creation of new child nodes with these candidates and their reflections builds out the search tree as described in the LATS framework.

# 5. Reflection Mechanism comparison:
# The reflection_chain in the notebook is a direct implementation of the LATS reflection step.
# - It uses an LLM specifically prompted to act as a critic.
# - It's designed to evaluate a given candidate response (or set of messages representing a step in the trajectory) based on criteria like sufficiency and quality.
# - The output is structured (using the Reflection Pydantic model), allowing for programmatic access to the critique and a quantitative score, which is essential for guiding the tree search (e.g., in UCB calculations and backpropagation).
# - This mechanism is applied both to the initial response and to the candidates generated during expansion, consistent with the iterative reflection process in LATS.

# 6. LangGraph Structure and Control Flow comparison:
# The LangGraph structure in the notebook closely matches the typical control flow of a LATS agent.
# - The use of 'start' and 'expand' nodes represents the main phases of the loop.
# - The conditional edges and the 'should_loop' function implement the core decision logic for continuing the search (expanding the tree further) or terminating (either because a solution is found or a search limit, like height, is reached).
# - This iterative structure of generating -> evaluating (reflecting) -> selecting/expanding -> evaluating (reflecting) is the fundamental cycle of LATS, and the LangGraph clearly orchestrates this flow.

# Based on this comparison, the notebook code demonstrates a clear and direct implementation of the core LATS framework as described in tutorials and literature, utilizing components and logic that closely mirror the conceptual steps of initial generation, iterative expansion and reflection guided by tree search principles.


## Identify differences

### Subtask:
Highlight areas where your implementation deviates from the tutorial, such as specific chains, node logic, graph structure, or tool usage.


**Reasoning**:
Analyze the notebook code to identify deviations from a standard LATS implementation based on the provided instructions. This involves comparing prompts, the Node class, selection/expansion logic, TreeState, graph structure, tool usage, and LLM parameters to typical LATS tutorial approaches.



In [ ]:
# 1. Prompts Comparison:
# - initial_answer_chain prompt: "You are an AI Assistant" + user input + optional messages. This is a fairly standard prompt for an initial generation step. Deviations would depend on the specific tutorial's initial prompt, but this one is general-purpose.
# - expansion_chain prompt: Uses the same prompt_template as the initial chain ("You are an AI Assistant" + user input + messages). The deviation here is not in the prompt template itself, but how it's used - it's fed the *trajectory* of messages from the selected node, which is standard for LATS expansion. The deviation might be in the *system instruction* compared to a tutorial that might give a more specific instruction for generating *diverse* candidates.
# - reflection_chain prompt: "Reflect and grade the assistant's response to the user question below" + user input + candidate messages. This prompt is specifically designed for self-reflection and grading, using the Reflection tool. This aligns closely with the LATS concept of having a dedicated reflection step. Any deviation would be in the specific wording or criteria requested for reflection compared to a tutorial. The notebook explicitly asks for critique on "sufficiency, superfluency, and general quality" and a "score from 0-10".

# 2. Node Class Implementation Deviations:
# - The Node class includes 'reflection' as a direct attribute, storing the Reflection Pydantic object. This tightly couples the node with its reflection, which is a specific design choice in this implementation. A basic MCTS node might just store the reward/score, while the reflection itself could be stored separately or generated on demand. This notebook's approach makes the reflection a first-class component of the node.
# - The '_is_solved' attribute and the '_mark_tree_as_solved' method explicitly track and propagate the "found_solution" status from the reflection up the tree. This is a specific mechanism for handling task completion within the tree structure, potentially a deviation from a pure MCTS where termination might solely rely on a value threshold or visit count.
# - The 'best_child_score' property specifically prioritizes solved nodes (`int(child.is_solved) * child.value`), which is a modification of a standard MCTS selection that might just use the raw UCB or value. This prioritizes branches that have found a solution.

# 3. Selection and Expansion Logic Deviations:
# - Selection ('select' function): Uses Upper Confidence Bound (UCB) as the selection criteria, which is standard for MCTS. The deviation could be in the specific 'exploration_weight' used (defaulting to 1.0 in the UCB formula within the Node class) compared to a tutorial, as this weight can significantly impact the exploration vs. exploitation balance.
# - Expansion ('expand' function): Generates 'N' candidates from the *single best* selected node's trajectory. Some LATS variations or tutorials might explore expanding from multiple promising nodes or use different strategies for generating candidates (e.g., prompting for diverse responses). The notebook's approach is a straightforward N-ary expansion from the single best leaf.
# - Tool Usage in Expansion: The notebook explicitly processes tool calls for *each* of the N generated candidates during expansion. This is a specific pattern of integrating tools into the expansion step, where each potential next message is evaluated for tool use and the tool results are incorporated *before* reflection. This might differ from tutorials where tool use might be a separate node after expansion or handled differently.

# 4. TreeState Deviations:
# - The TreeState is simple, containing just 'root' and 'input'. This is likely standard for many LATS implementations. Any deviation would depend on whether a tutorial included additional state variables (e.g., a global best score found so far, a list of all terminal nodes).

# 5. LangGraph Structure and should_loop Deviations:
# - Graph Structure: A simple two-node loop ('start' -> 'expand' -> 'expand'...) is used. This is a common pattern for iterative processes like LATS.
# - should_loop logic: The termination conditions are based on `root.is_solved` and `root.height > 5`. The height limit (5) is a specific, potentially arbitrary, deviation from a tutorial that might use a different maximum depth or a different termination condition altogether (e.g., a score threshold, a maximum number of iterations, or no explicit depth limit). The edge names ("expand", END) are functional and likely standard.

# 6. Tool Usage Integration Deviations:
# - Tools are integrated at two key stages: initial generation and expansion. In both cases, the LLM is bound with tools, the response is parsed for tool calls, and the tool_node executes them, incorporating the results into the messages *before* reflection. This specific pattern of tool execution and message integration prior to reflection is a notable aspect of this implementation. A deviation would be if a tutorial handled tool use as a separate step *after* reflection or used a different mechanism for integrating tool results.

# 7. LLM Parameters and Model Deviations:
# - LLM Model: Uses 'moonshotai/kimi-k2:free' via OpenRouter. The specific model choice is a clear deviation from a tutorial that might use OpenAI's standard models (like gpt-4 or gpt-3.5-turbo) or other providers. The choice of a free model might impact performance and capabilities compared to models typically used in tutorials.
# - `llm.generate(n=n)` in expansion: Explicitly sets `n` (defaulting to 5 via config) to generate multiple candidates. The value of N is a hyperparameter choice that could deviate from a tutorial's default or recommended value. This directly controls the breadth of the search tree expansion at each step.

# Overall, the notebook implements the core LATS concepts faithfully but introduces specific design choices, particularly around the tight coupling of Nodes with Reflections, the explicit tracking of a solved state, the specific termination conditions (height limit), the pattern of tool integration before reflection, and the choice of LLM and N value.

## Summarize the comparison

### Subtask:
Provide a summary of the key similarities and differences found between your code and the tutorial's code.


## Summary:

### Data Analysis Key Findings

*   The notebook implements the core LATS concepts of initial generation, expansion, reflection, and tree search logic using LangGraph.
*   The `Node` class in the notebook includes attributes like `messages`, `parent`/`children`, `value`, `visits`, and importantly, `reflection`, aligning with LATS node requirements. It also includes methods for UCB calculation, backpropagation, and trajectory retrieval.
*   The `TreeState` is a simple `TypedDict` holding the `root` node and the original `input`, representing the state of the search tree.
*   Dedicated chains (`initial_answer_chain`, `expansion_chain`, `reflection_chain`) are used for the key LATS steps, leveraging `ChatPromptTemplate` and tool binding.
*   Tool usage (specifically `TavilySearch`) is integrated into both the initial generation and expansion steps.
*   Reflection is implemented as a distinct chain that uses a dedicated `Reflection` tool and `PydanticToolsParser` to evaluate candidate responses and assign a score.
*   The LangGraph structure uses `start` and `expand` nodes in an iterative loop.
*   The graph's control flow is managed by a `should_loop` function that terminates based on whether the root node is marked as solved or if the tree height exceeds a limit (hardcoded at 5).
*   Deviations from a standard LATS tutorial might include the tight coupling of `reflection` within the `Node` class, the explicit `_is_solved` tracking, the modification of `best_child_score` to prioritize solved nodes, the specific strategy of expanding from only the single best node, the hardcoded height limit for termination, the consistent pattern of executing tools *before* reflection, and the choice of a specific non-OpenAI LLM and the explicit setting of the `n` parameter for candidate generation.

### Insights or Next Steps

*   The notebook provides a functional, albeit specific, implementation of the LATS framework in LangGraph, demonstrating the core principles.
*   Future work could involve making the tree height limit and exploration weight configurable parameters, exploring different node selection strategies, or implementing alternative expansion methods (e.g., expanding from multiple nodes).
